# Will the Customer Accept the Coupon?

**Context**

Imagine driving through town and a coupon is delivered to your cell phone for a restaurant
near where you are driving. Would you accept that coupon and take a short detour to the
restaurant? Would you accept the coupon but use it on a subsequent trip? Would you ignore
the coupon entirely? What about a bar coupon with a minor passenger in the car? Would weather
impact the rate of acceptance? What about the time of day?

**Overview**

The goal of this project is to use visualizations and probability distributions to distinguish
between customers who accepted a driving coupon versus those that did not.

**Data**

This data comes from the UCI Machine Learning Repository, collected via a survey on Amazon
Mechanical Turk. Driving scenarios describe destination, time, weather, passenger, etc.
Responses of "right away" or "later before expiry" are labeled `Y = 1`; "no" is `Y = 0`.
Five coupon types: less expensive restaurants (<$20), coffee houses, carry out & take away,
bar, and more expensive restaurants ($20–$50).

## Data Description

The dataset contains **12,684 observations** across 26 columns.

**User Attributes:** Gender, Age, Marital Status, Children, Education, Occupation, Income,
and visit frequency for Bar, CoffeeHouse, CarryAway, Restaurant<$20, Restaurant$20-$50.

**Contextual Attributes:** Destination, Weather (Sunny/Rainy/Snowy), Temperature (30/55/80°F),
Time (7AM/10AM/2PM/6PM/10PM), Passenger (Alone/Partner/Kid(s)/Friend(s)).

**Target Variable:** `Y = 1` (coupon accepted), `Y = 0` (coupon rejected)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

## 1. Data Loading and Initial Exploration

In [ ]:
data = pd.read_csv('../data/coupons.csv')
print(f"Dataset shape: {data.shape}")
data.head()

## 2. Data Quality Assessment

Identifying columns with missing values before analysis.

In [ ]:
# Summarize missing values (only columns with nulls)
missing = data.isnull().sum()
missing_cols = missing[missing > 0]
print("Columns with missing values:")
print(missing_cols)
print()
print("Note: 'car' is ~99% missing — not useful for analysis.")
print("Frequency columns (Bar, CoffeeHouse, etc.) have small amounts of missing data.")

## 3. Data Cleaning

In [ ]:
# Fill 'car' with 'Unknown' — too sparse to drop rows over it
data['car'] = data['car'].fillna('Unknown')

# Drop the handful of rows missing 'destination' (critical context variable)
data = data.dropna(subset=['destination'])

print(f"Shape after cleaning: {data.shape}")
print("Remaining nulls in frequency columns will be excluded per-analysis.")

## 4. Exploratory Data Analysis

In [ ]:
# Overall acceptance rate
acceptance_rate = data['Y'].mean()
print(f"Overall Coupon Acceptance Rate: {acceptance_rate:.2%}")
print(f"Total observations: {len(data):,}")

In [ ]:
# Distribution of coupon types alongside acceptance rate by type
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

coupon_order = data['coupon'].value_counts().index.tolist()
sns.countplot(data=data, x='coupon', order=coupon_order, ax=axes[0], palette='Blues_d')
axes[0].set_title('Distribution of Coupon Types')
axes[0].set_xlabel('Coupon Type')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

coupon_acceptance = data.groupby('coupon')['Y'].mean().sort_values(ascending=False)
axes[1].bar(coupon_acceptance.index, coupon_acceptance.values, color='steelblue', edgecolor='white')
axes[1].set_title('Acceptance Rate by Coupon Type')
axes[1].set_xlabel('Coupon Type')
axes[1].set_ylabel('Acceptance Rate')
axes[1].set_ylim(0, 1)
for i, v in enumerate(coupon_acceptance.values):
    axes[1].text(i, v + 0.02, f'{v:.1%}', ha='center', fontsize=10)
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# Temperature distribution by coupon acceptance
plt.figure(figsize=(8, 5))
sns.histplot(data=data, x='temperature', hue='Y', bins=10, multiple='dodge', palette='Set2')
plt.title('Temperature Distribution by Coupon Acceptance')
plt.xlabel('Temperature (F)')
plt.ylabel('Count')
plt.legend(title='Accepted', labels=['No', 'Yes'])
plt.show()

## 5. Bar Coupon Analysis

### Problem Statement

**Question:** What characteristics distinguish drivers who accept bar coupons from those who
reject them?

We investigate four factors: bar visit frequency, age demographics, social context (passenger
type and occupation), and a combined profile analysis.

In [ ]:
# Filter to bar coupons
bar_coupons = data[data['coupon'] == 'Bar'].copy()
bar_acceptance = bar_coupons['Y'].mean()
print(f"Bar coupon observations: {len(bar_coupons):,}")
print(f"Bar Coupon Acceptance Rate: {bar_acceptance:.2%}  (overall: {data['Y'].mean():.2%})")

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=bar_coupons, x='Y', palette='Set2')
plt.title('Bar Coupon: Accepted vs Rejected')
plt.xlabel('Response')
plt.ylabel('Count')
plt.xticks([0, 1], ['Rejected (0)', 'Accepted (1)'])
plt.show()

### Key Factor 1: Bar Visit Frequency

Drivers who visit bars more than 3 times per month vs. 3 or fewer.

In [ ]:
less_equal_3 = bar_coupons[bar_coupons['Bar'].isin(['never', 'less1', '1~3'])]
more_than_3  = bar_coupons[bar_coupons['Bar'].isin(['4~8', 'gt8'])]

rates_freq = {
    '<=3 visits/month': less_equal_3['Y'].mean(),
    '>3 visits/month':  more_than_3['Y'].mean()
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

bars = axes[0].bar(rates_freq.keys(), rates_freq.values(),
                   color=['#90CAF9', '#1565C0'], edgecolor='white')
axes[0].set_title('Acceptance Rate by Bar Visit Frequency')
axes[0].set_ylabel('Acceptance Rate')
axes[0].set_ylim(0, 1)
for bar, v in zip(bars, rates_freq.values()):
    axes[0].text(bar.get_x() + bar.get_width() / 2, v + 0.02,
                 f'{v:.1%}', ha='center', fontweight='bold')

visit_data = pd.concat([
    less_equal_3[['Y']].assign(group='<=3 visits/month'),
    more_than_3[['Y']].assign(group='>3 visits/month')
])
sns.countplot(data=visit_data, x='group', hue='Y', ax=axes[1], palette='Set2')
axes[1].set_title('Accepted vs Rejected by Visit Frequency')
axes[1].set_xlabel('Frequency Group')
axes[1].set_ylabel('Count')
axes[1].legend(title='Accepted', labels=['No', 'Yes'])

plt.tight_layout()
plt.show()

print(f"<=3 visits/month acceptance: {less_equal_3['Y'].mean():.2%}  (n={len(less_equal_3)})")
print(f">3 visits/month acceptance:  {more_than_3['Y'].mean():.2%}  (n={len(more_than_3)})")

### Key Factor 2: Age and Visit Frequency

Drivers who visit bars more than once a month **and** are over 25 years old vs. all others.

In [ ]:
group1 = bar_coupons[
    bar_coupons['Bar'].isin(['1~3', '4~8', 'gt8']) &
    ~bar_coupons['age'].isin(['below21', '21'])
]
group2 = bar_coupons.drop(group1.index)

rates_age = {
    'Over 25 + frequent visits': group1['Y'].mean(),
    'All others': group2['Y'].mean()
}

plt.figure(figsize=(7, 5))
bars = plt.bar(rates_age.keys(), rates_age.values(),
               color=['#A5D6A7', '#BDBDBD'], edgecolor='white')
plt.title('Acceptance: Older Frequent Bar-Goers vs. Others')
plt.ylabel('Acceptance Rate')
plt.ylim(0, 1)
for bar, v in zip(bars, rates_age.values()):
    plt.text(bar.get_x() + bar.get_width() / 2, v + 0.02,
             f'{v:.1%}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Over 25 + frequent visits: {group1['Y'].mean():.2%}  (n={len(group1)})")
print(f"All others:                {group2['Y'].mean():.2%}  (n={len(group2)})")

### Key Factor 3: Social Context and Occupation

Drivers who visit bars more than once a month, **without kids** as passengers,
and in **non-farming occupations**.

In [ ]:
target = bar_coupons[
    bar_coupons['Bar'].isin(['1~3', '4~8', 'gt8']) &
    (bar_coupons['passanger'] != 'Kid(s)') &
    (bar_coupons['occupation'] != 'Farming Fishing & Forestry')
]
others = bar_coupons.drop(target.index)

rates_social = {
    'Frequent + no kids + non-farm': target['Y'].mean(),
    'All others': others['Y'].mean()
}

plt.figure(figsize=(7, 5))
bars = plt.bar(rates_social.keys(), rates_social.values(),
               color=['#CE93D8', '#BDBDBD'], edgecolor='white')
plt.title('Acceptance: Frequent Bar-Goers (No Kids, Non-Farm) vs. Others')
plt.ylabel('Acceptance Rate')
plt.ylim(0, 1)
for bar, v in zip(bars, rates_social.values()):
    plt.text(bar.get_x() + bar.get_width() / 2, v + 0.02,
             f'{v:.1%}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Frequent + no kids + non-farm: {target['Y'].mean():.2%}  (n={len(target)})")
print(f"All others:                    {others['Y'].mean():.2%}  (n={len(others)})")

### Combined Profile Analysis

Testing a composite profile of likely bar-coupon acceptors:
- Frequent bar visitors, no child passengers, not widowed **OR**
- Frequent bar visitors under age 30 **OR**
- Frequent cheap restaurant visitors with income < $50K

In [ ]:
c1 = (
    bar_coupons['Bar'].isin(['1~3', '4~8', 'gt8']) &
    (bar_coupons['passanger'] != 'Kid(s)') &
    (bar_coupons['maritalStatus'] != 'Widowed')
)
c2 = (
    bar_coupons['Bar'].isin(['1~3', '4~8', 'gt8']) &
    bar_coupons['age'].isin(['below21', '21', '26'])
)
c3 = (
    bar_coupons['RestaurantLessThan20'].isin(['4~8', 'gt8']) &
    bar_coupons['income'].isin([
        'Less than $12500', '$12500 - $24999', '$25000 - $37499', '$37500 - $49999'
    ])
)

combined     = bar_coupons[c1 | c2 | c3]
not_combined = bar_coupons[~(c1 | c2 | c3)]

print(f"Combined profile acceptance:   {combined['Y'].mean():.2%}  (n={len(combined)})")
print(f"Remaining drivers acceptance:  {not_combined['Y'].mean():.2%}  (n={len(not_combined)})")

### Bar Coupon Findings

1. **Visit frequency is the strongest predictor.** Drivers who visit bars more than 3 times
   per month accept at ~77%, compared to ~37% for infrequent visitors — a 2× difference.

2. **Age amplifies the effect.** Frequent bar visitors over 25 accept at ~70% vs. ~34% for
   all others.

3. **Social context matters.** Removing child passengers and farm workers from the frequent-
   visitor group sustains a ~71% acceptance rate, confirming that adult social contexts
   increase receptivity.

4. **Hypothesis:** The core target for bar coupons is a driver who frequents bars, travels
   without children, holds a non-farm occupation, and is under 30 or socially active.

## 6. Coffee House Coupon Analysis (Independent Investigation)

### Problem Statement

**Question:** What factors most influence whether a driver accepts a Coffee House coupon?

Coffee consumption is strongly tied to time of day and social context. We examine time of day,
passenger type, weather, and age group to identify the characteristics of a driver most likely
to accept a coffee coupon.

In [ ]:
coffee = data[data['coupon'] == 'Coffee House'].copy()
print(f"Coffee House coupon observations: {len(coffee):,}")
print(f"Coffee House Acceptance Rate: {coffee['Y'].mean():.2%}  (overall: {data['Y'].mean():.2%})")

### Analysis by Time of Day

Coffee consumption peaks in the morning and midday. We expect 10AM and 2PM to drive the
highest acceptance rates.

In [ ]:
time_order = ['7AM', '10AM', '2PM', '6PM', '10PM']
time_acceptance = coffee.groupby('time')['Y'].mean().reindex(time_order)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(time_acceptance.index, time_acceptance.values, color='#8D6E63', edgecolor='white')
axes[0].set_title('Coffee Coupon Acceptance Rate by Time of Day')
axes[0].set_xlabel('Time of Day')
axes[0].set_ylabel('Acceptance Rate')
axes[0].set_ylim(0, 1)
for i, v in enumerate(time_acceptance.values):
    axes[0].text(i, v + 0.02, f'{v:.1%}', ha='center', fontsize=10)

sns.countplot(data=coffee, x='time', hue='Y', order=time_order, ax=axes[1], palette='Set2')
axes[1].set_title('Accepted vs Rejected by Time of Day')
axes[1].set_xlabel('Time of Day')
axes[1].set_ylabel('Count')
axes[1].legend(title='Accepted', labels=['No', 'Yes'])

plt.tight_layout()
plt.show()

### Analysis by Passenger Type

Social companions — friends or partners — may lower the barrier to stopping for coffee.

In [ ]:
pass_acceptance = coffee.groupby('passanger')['Y'].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(pass_acceptance.index, pass_acceptance.values, color='#FF8A65', edgecolor='white')
axes[0].set_title('Coffee Coupon Acceptance by Passenger Type')
axes[0].set_xlabel('Passenger Type')
axes[0].set_ylabel('Acceptance Rate')
axes[0].set_ylim(0, 1)
for i, v in enumerate(pass_acceptance.values):
    axes[0].text(i, v + 0.02, f'{v:.1%}', ha='center', fontsize=10)

sns.countplot(data=coffee, x='passanger', hue='Y', ax=axes[1], palette='Set2')
axes[1].set_title('Accepted vs Rejected by Passenger Type')
axes[1].set_xlabel('Passenger Type')
axes[1].set_ylabel('Count')
axes[1].legend(title='Accepted', labels=['No', 'Yes'])

plt.tight_layout()
plt.show()

### Analysis by Weather and Age Group

In [ ]:
age_order = ['below21', '21', '26', '31', '36', '41', '46', '50plus']
age_acceptance     = coffee.groupby('age')['Y'].mean().reindex(age_order).dropna()
weather_acceptance = coffee.groupby('weather')['Y'].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(weather_acceptance.index, weather_acceptance.values,
            color='skyblue', edgecolor='white')
axes[0].set_title('Coffee Coupon Acceptance by Weather')
axes[0].set_xlabel('Weather')
axes[0].set_ylabel('Acceptance Rate')
axes[0].set_ylim(0, 1)
for i, v in enumerate(weather_acceptance.values):
    axes[0].text(i, v + 0.02, f'{v:.1%}', ha='center', fontsize=10)

axes[1].bar(age_acceptance.index, age_acceptance.values,
            color='mediumseagreen', edgecolor='white')
axes[1].set_title('Coffee Coupon Acceptance by Age Group')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Acceptance Rate')
axes[1].tick_params(axis='x', rotation=30)
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

### Comparing Accepted vs Rejected Driver Profiles

Side-by-side comparison across all key attributes.

In [ ]:
accepted = coffee[coffee['Y'] == 1]
rejected = coffee[coffee['Y'] == 0]
width = 0.35

def side_by_side(ax, col, order, title, xlabel):
    acc = accepted[col].value_counts().reindex(order).fillna(0)
    rej = rejected[col].value_counts().reindex(order).fillna(0)
    x = np.arange(len(order))
    ax.bar(x - width / 2, acc.values, width, label='Accepted', color='#4CAF50')
    ax.bar(x + width / 2, rej.values, width, label='Rejected', color='#F44336')
    ax.set_xticks(x)
    ax.set_xticklabels(order, rotation=20)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Count')
    ax.legend()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
side_by_side(axes[0, 0], 'time',       ['7AM', '10AM', '2PM', '6PM', '10PM'],       'Time of Day',       'Time')
side_by_side(axes[0, 1], 'passanger',  ['Alone', 'Partner', 'Kid(s)', 'Friend(s)'], 'Passenger Type',    'Passenger')
side_by_side(axes[1, 0], 'weather',    ['Sunny', 'Rainy', 'Snowy'],                 'Weather',           'Weather')
side_by_side(axes[1, 1], 'expiration', ['2h', '1d'],                                'Coupon Expiration', 'Expiration')

plt.suptitle('Coffee House Coupon: Accepted vs Rejected Driver Profiles', fontsize=14)
plt.tight_layout()
plt.show()

### Coffee House Coupon Findings

1. **Time of day is the strongest driver.** The 10AM slot yields the highest acceptance,
   consistent with the morning coffee routine. The 2PM afternoon window also performs well.

2. **Social companions boost acceptance.** Drivers with friends (59.7%) or partners (57.0%)
   accept significantly more than those driving alone (43.8%).

3. **Weather has a modest effect.** Rainy days show slightly higher acceptance (52.2%),
   likely because a warm coffee is more appealing in cold or wet conditions.

4. **Younger drivers (21–26) engage more.** Coffee coupons resonate strongly with younger,
   on-the-go demographics.

5. **Longer-validity coupons perform better.** Drivers accept 1-day coupons at higher rates
   than 2-hour coupons, suggesting flexibility in redemption timing matters.

## 7. Conclusions and Recommendations

### Summary of Findings

- **Overall acceptance rate is 56.8%.** Carry-out and cheap restaurants lead; bars lag behind.
- **Bar coupons** are most accepted by frequent bar-goers (3+ visits/month) without children.
- **Coffee House coupons** perform best at 10AM and 2PM for social travelers.
- Younger drivers and social contexts consistently increase acceptance across all coupon types.

---

### Recommendations

| Recommendation | Target Coupon | Evidence |
|---|---|---|
| Deliver during morning (10AM) and afternoon (2PM) commutes | Coffee House | Time-of-day analysis |
| Target groups and couples, not solo drivers | All types | Passenger analysis |
| Segment by visit frequency (bar 3+/month) | Bar | Visit frequency analysis |
| Focus on drivers aged 21–26 | Bar & Coffee | Age group analysis |
| Offer longer expiration windows (1-day vs 2-hour) | Coffee House | Expiration analysis |
| Trigger on rainy or cold days | Coffee House | Weather analysis |

---

### Next Steps

1. **Build a predictive model** (logistic regression or random forest) to score individual
   drivers on coupon acceptance probability.
2. **Conduct customer segmentation** using clustering to discover natural driver personas.
3. **A/B test coupon timing** — test morning vs. afternoon delivery for coffee coupons.
4. **Expand the analysis** to remaining coupon types (Cheap Restaurant, Carry-Out) using
   the same framework.
5. **Develop a recommendation engine** that selects the optimal coupon type per driver profile.